# MVNX Parquet 시계열 데이터 검증 노트북
이 노트북은 전처리된 Parquet 파일들의 무결성, 컬럼 구조, 메타데이터 주입 여부를 자동으로 탐색하고 리포팅하는 재사용 가능한 함수 컴포넌트들을 제공합니다.

In [1]:
import pandas as pd
from pathlib import Path
import os

In [2]:
def verify_parquet_file(file_path):
    """
    단일 Parquet 파일을 열어 메타데이터 및 시계열 결측치, Shape를 검증합니다.
    """
    path_obj = Path(file_path)
    print(f"\n{'='*50}")
    print(f"🔍 [분석 시작]: {path_obj.name}")
    
    # 1. 파일 로딩 속도 및 Shape
    df = pd.read_parquet(path_obj)
    print(f"✅ Data Shape: {df.shape[0]:,} Rows × {df.shape[1]:,} Columns")
    
    # 2. 필수 메타데이터 컬럼 검증
    meta_cols = ['time_ms', 'group', 'subject_id', 'speed', 'file_name']
    missing_meta = [c for c in meta_cols if c not in df.columns]
    if missing_meta:
        print(f"⚠️ 누락된 메타데이터 컬럼: {missing_meta}")
    else:
        print(f"✅ 필수 메타데이터({len(meta_cols)}종) 정상 마운트")
        
    # 3. 고유 속성군 분포 (Subject, Speed 종류 파악)
    if 'subject_id' in df.columns:
        print(f"👤 포함된 피험자 종류({df['subject_id'].nunique()}명): {df['subject_id'].unique()[:5].tolist()}...")
    if 'speed' in df.columns:
        print(f"🏃‍♂️ 포함된 보행속도 분포: {df['speed'].unique().tolist()}")
        
    # 4. 시계열 데이터(Features) 결측값(NaN) 요약
    feature_cols = [c for c in df.columns if c not in meta_cols]
    na_counts = df[feature_cols].isna().sum()
    cols_with_na = na_counts[na_counts > 0]
    if len(cols_with_na) > 0:
        print(f"⚠️ 결측치(NaN)가 포함된 피처 개수: {len(cols_with_na)}개")
    else:
        print("✅ 시계열 부분 결측치 완전 무결 (Clean Data)")
        
    return df

In [3]:

# 노트북 위치를 기반으로 상대경로 설정 (Soft-coding)
PROCESSED_DIR = Path("../data/processed").resolve()

# 🪣 1. 빈 데이터 '리스트' 바구니를 미리 준비합니다 (빈 데이터프레임 X)
df_list = []

if not PROCESSED_DIR.exists():
    print(f"경로를 찾을 수 없습니다: {PROCESSED_DIR}")
else:
    parquet_files = list(PROCESSED_DIR.glob("*.parquet"))
    parquet_files = [f for f in parquet_files if "Master_Gait_Dataset" not in f.name and "raw_merged" not in f.name]

    print(f"총 {len(parquet_files)}개의 데이터 병합 파케이(Parquet) 파일 발견됨.\n")
    
    # 전체 파일 대상 구조 분해 및 검증 함수 반복 실행
    for p_file in parquet_files:
        df = verify_parquet_file(p_file) # 검증 함수가 df를 반환함
        df_list.append(df) # 🪣 2. 리스트 바구니에 로드된 df를 하나씩 가볍게 던져 넣음!

    # 🔨 3. 루프가 모두 끝난 뒤, 바구니에 담긴 4개의 덩어리를 단 한 번의 연산으로 합체!
    print("전체 데이터를 메모리에서 하나로 결합하는 중...")
    merged_df = pd.concat(df_list, ignore_index=True)
    
    print(f"\n✅ 드디어 하나로 차곡차곡 쌓인 마스터 데이터프레임 완성!")
    display(merged_df.info())  # 결과 확인


총 4개의 데이터 병합 파케이(Parquet) 파일 발견됨.


🔍 [분석 시작]: ACLD.parquet
✅ Data Shape: 510,051 Rows × 693 Columns
✅ 필수 메타데이터(5종) 정상 마운트
👤 포함된 피험자 종류(40명): ['ACLD9', 'ACLD8', 'ACLD7', 'ACLD6', 'ACLD5']...
🏃‍♂️ 포함된 보행속도 분포: ['fast', 'normal', 'slow']
⚠️ 결측치(NaN)가 포함된 피처 개수: 70개

🔍 [분석 시작]: ACLR.parquet
✅ Data Shape: 311,449 Rows × 693 Columns
✅ 필수 메타데이터(5종) 정상 마운트
👤 포함된 피험자 종류(27명): ['ACLR9', 'ACLR8', 'ACLR4', 'ACLR38', 'ACLR37']...
🏃‍♂️ 포함된 보행속도 분포: ['normal', 'fast', 'slow']
⚠️ 결측치(NaN)가 포함된 피처 개수: 70개

🔍 [분석 시작]: Healthy adolescents.parquet
✅ Data Shape: 282,954 Rows × 693 Columns
✅ 필수 메타데이터(5종) 정상 마운트
👤 포함된 피험자 종류(22명): ['HK9', 'HK8', 'HK7', 'HK6', 'HK5']...
🏃‍♂️ 포함된 보행속도 분포: ['fast', 'normal', 'slow']
✅ 시계열 부분 결측치 완전 무결 (Clean Data)

🔍 [분석 시작]: Healthy adults.parquet
✅ Data Shape: 332,902 Rows × 693 Columns
✅ 필수 메타데이터(5종) 정상 마운트
👤 포함된 피험자 종류(25명): ['HA9', 'HA8', 'HA7', 'HA6', 'HA5']...
🏃‍♂️ 포함된 보행속도 분포: ['normal', 'fast', 'slow']
⚠️ 결측치(NaN)가 포함된 피처 개수: 70개
전체 데이터를 메모리에서 하나로 결합하는 중...

✅ 드디어 하나로

None

In [4]:
display(merged_df.info())
display(merged_df.head())
display(merged_df.describe())
output_path = PROCESSED_DIR / "raw_merged.parquet"

merged_df.to_parquet(output_path, index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1437356 entries, 0 to 1437355
Columns: 693 entries, time_ms to sensorOrientation_27
dtypes: float64(688), object(5)
memory usage: 7.4+ GB


None

,time_ms,orientation_0,orientation_1,orientation_2,orientation_3,orientation_4,orientation_5,orientation_6,orientation_7,orientation_8,...,sensorOrientation_18,sensorOrientation_19,sensorOrientation_20,sensorOrientation_21,sensorOrientation_22,sensorOrientation_23,sensorOrientation_24,sensorOrientation_25,sensorOrientation_26,sensorOrientation_27
0,1653819322093,0.998987,0.002905,-0.044690,-0.004504,0.997571,0.002792,-0.069455,-0.004575,0.991368,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1653819322103,0.998996,0.002682,-0.044552,-0.003946,0.997583,0.002583,-0.069317,-0.004012,0.991389,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1653819322113,0.999005,0.002451,-0.044404,-0.003397,0.997596,0.002366,-0.069169,-0.003457,0.991411,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1653819322123,0.999013,0.002225,-0.044261,-0.002843,0.997608,0.002153,-0.069026,-0.002898,0.991432,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1653819322133,0.999021,0.002012,-0.044138,-0.002273,0.997619,0.001955,-0.068903,-0.002322,0.991450,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,orientation_0,orientation_1,orientation_2,orientation_3,orientation_4,orientation_5,orientation_6,orientation_7,orientation_8,orientation_9,...,sensorOrientation_18,sensorOrientation_19,sensorOrientation_20,sensorOrientation_21,sensorOrientation_22,sensorOrientation_23,sensorOrientation_24,sensorOrientation_25,sensorOrientation_26,sensorOrientation_27
count,1.437356e+06,1.437356e+06,1.437356e+06,1.437356e+06,1.437356e+06,1.437356e+06,1.437356e+06,1.437356e+06,1.437356e+06,1.437356e+06,...,1.009450e+06,1.009450e+06,1.009450e+06,1.009450e+06,1.009450e+06,1.009450e+06,1.009450e+06,1.009450e+06,1.009450e+06,1.009450e+06
mean,9.904357e-01,-7.826888e-04,1.909505e-02,3.671098e-02,9.906047e-01,1.280365e-04,-5.476699e-03,3.672188e-02,9.883707e-01,2.398378e-03,...,-5.530987e-01,-4.283445e-01,1.496738e-01,1.900236e-01,-1.110455e-01,1.466411e-01,1.937602e-01,-1.932031e-01,-1.055740e-01,-5.289395e-01
std,3.807036e-02,2.073197e-02,4.198139e-02,1.169723e-01,3.804503e-02,2.036133e-02,4.200433e-02,1.170365e-01,3.800369e-02,2.116117e-02,...,9.691416e-02,1.190573e-01,1.091173e-01,7.421125e-01,1.601556e-01,5.588946e-01,2.011360e-01,2.836416e-01,1.740855e-01,6.897724e-01
min,-1.488810e-01,-1.557920e-01,-1.780320e-01,-9.999640e-01,-1.485050e-01,-1.556470e-01,-2.023770e-01,-9.999840e-01,-1.471680e-01,-2.009650e-01,...,-8.437690e-01,-8.077510e-01,-1.365140e-01,-9.848200e-01,-9.626340e-01,-8.896660e-01,-1.962920e-01,-8.927620e-01,-7.523370e-01,-9.997890e-01
25%,9.911610e-01,-1.361100e-02,-6.945000e-03,-2.839300e-02,9.913860e-01,-1.252200e-02,-3.154500e-02,-2.840600e-02,9.882500e-01,-1.096425e-02,...,-6.203590e-01,-5.070390e-01,6.799500e-02,-7.206537e-01,-2.023690e-01,-4.975348e-01,7.034625e-02,-3.122095e-01,-2.183277e-01,-9.461070e-01
50%,9.962730e-01,-8.670000e-04,2.026100e-02,3.403750e-02,9.964020e-01,-1.370000e-04,-4.309000e-03,3.405600e-02,9.937370e-01,1.852000e-03,...,-5.586510e-01,-4.337515e-01,1.344480e-01,6.699745e-01,-1.194955e-01,4.224775e-01,1.431000e-01,-2.116165e-01,-1.504080e-01,-9.052295e-01
75%,9.984570e-01,1.178700e-02,4.554425e-02,9.989900e-02,9.985750e-01,1.260900e-02,2.096500e-02,9.995225e-02,9.968170e-01,1.566900e-02,...,-4.919952e-01,-3.598270e-01,2.086340e-01,7.804410e-01,-3.239300e-02,6.242130e-01,2.391370e-01,-7.734225e-02,-4.086000e-02,-5.944118e-01
max,1.000000e+00,1.821800e-01,2.129270e-01,9.999720e-01,1.000000e+00,1.754700e-01,1.886520e-01,9.999980e-01,9.999990e-01,1.594060e-01,...,5.159020e-01,7.584200e-01,8.385170e-01,9.894570e-01,9.677880e-01,8.902780e-01,9.875980e-01,8.727920e-01,8.598100e-01,9.999450e-01


In [6]:
import pandas as pd

# 화면에 출력할 행(Row, 세로줄) 개수에 제한을 두지 않음
pd.set_option('display.max_rows', None)

# 화면에 출력할 열(Column, 가로줄) 개수에 제한을 두지 않음
pd.set_option('display.max_columns', None)

# 한 셀(칸) 안에 들어가는 글자 수가 길어도 절대 자르지 않음
pd.set_option('display.max_colwidth', None)



# 전체 행 개수 대비 결측치 비율(%) 계산
na_percent = (df.isna().mean() * 100)
bad_cols_percent = na_percent[na_percent > 0].sort_values(ascending=False)

print("결측도 상위 불량 컬럼 목록(%):")
display(bad_cols_percent.head(100)) # 상위 20개만 보기


결측도 상위 불량 컬럼 목록(%):


sensorFreeAcceleration_0     33.341944
sensorFreeAcceleration_1     33.341944
sensorFreeAcceleration_2     33.341944
sensorFreeAcceleration_3     33.341944
sensorFreeAcceleration_4     33.341944
sensorFreeAcceleration_5     33.341944
sensorFreeAcceleration_6     33.341944
sensorFreeAcceleration_7     33.341944
sensorFreeAcceleration_8     33.341944
sensorFreeAcceleration_9     33.341944
sensorFreeAcceleration_10    33.341944
sensorFreeAcceleration_11    33.341944
sensorFreeAcceleration_12    33.341944
sensorFreeAcceleration_13    33.341944
sensorFreeAcceleration_14    33.341944
sensorFreeAcceleration_15    33.341944
sensorFreeAcceleration_16    33.341944
sensorFreeAcceleration_17    33.341944
sensorFreeAcceleration_18    33.341944
sensorFreeAcceleration_19    33.341944
sensorFreeAcceleration_20    33.341944
sensorMagneticField_0        33.341944
sensorMagneticField_1        33.341944
sensorMagneticField_2        33.341944
sensorMagneticField_3        33.341944
sensorMagneticField_4    